# Lab 1 · LLM Calls: the atom of everything

Every LangChain app, from a one-line helper to a fleet of coordinated agents, bottoms out in one act: a call to a model. Get the atom right and the molecules assemble themselves.

**What you build, and run, in this lab**

| You will | With |
|---|---|
| Send a structured call to a model on Bedrock | `ChatBedrockConverse` |
| Read replies as typed parts, not one blob | `content_blocks` |
| Template inputs instead of gluing strings | `ChatPromptTemplate` |
| Snap steps into one runnable object | the LCEL pipe operator |
| Get Python objects back instead of prose | `with_structured_output` |
| Swap providers with a single line | `init_chat_model` |

**The anchor.** Every example serves one running story: *TravelMind*, an airline support assistant. Booking `JX48Q2`, passenger Rao, Gold tier, the BLR to DEL leg cancelled. Same scenario, growing capability, lab after lab. You are never learning an API in the abstract; you are shipping one assistant, one layer at a time.

## Where "the call" sits in the LangChain universe

Read this map once. The whole library is this picture at different zoom levels.

```mermaid
graph TD
    PT["Prompt template"] --> MSG["Messages: system, human, ai"]
    MSG --> MODEL["Chat model: ChatBedrockConverse"]
    MODEL --> CB["Reply: content_blocks"]
    CB --> PARSE["Structured output: Pydantic object"]
    MODEL --> PIPE["LCEL pipe: prompt then model then parser"]
    PIPE --> UP["Built on top of this atom: workflows, agents, graphs"]
```

The dashed truth: workflows (Lab 2), agents (Lab 3), and multi-agent systems (Lab 4) are all just this atom, called in smarter arrangements.

## Setup

| Need | Value |
|---|---|
| Packages | `langchain`, `langchain-aws`, `langchain-core`, `pydantic` |
| Model access | Bedrock model access enabled in the account |
| Region | `us-east-1` |
| Model id | `us.anthropic.claude-haiku-4-5-20251001-v1:0` |
| Auth | AWS credentials reachable by boto3 (env vars, profile, or role) |

The `us.` prefix on the model id is not decoration. It selects a cross-region inference profile, which Bedrock requires for these models. Drop it and the call fails.

In [ ]:
# Run once per environment
# %pip install -U langchain langchain-aws langchain-core pydantic

In [1]:
from langchain_aws import ChatBedrockConverse

MODEL_ID = "us.anthropic.claude-haiku-4-5-20251001-v1:0"
REGION = "us-east-1"

llm = ChatBedrockConverse(
    model_id=MODEL_ID,
    region_name=REGION,
    temperature=0.2,
)
llm

ChatBedrockConverse(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11', 'langchain-aws': '1.6.1'}}, output_version=None, profile={'name': 'Claude Haiku 4.5', 'release_date': '2025-10-15', 'last_updated': '2025-10-15', 'open_weights': False, 'max_input_tokens': 200000, 'max_output_tokens': 64000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True}, client=<botocore.client.BedrockRuntime object at 0x110a1aa50>, bedrock_client=<botocore.client.Bedrock object at 0x110a1a900>, model_id='us.anthropic.claude-haiku-4-5-20251001-v1:0', temperature=0.2, region_name='us-east-1', aws_access_key_id=None, aws_secret_access_key=None, aw

## A call is three moves

Request goes in, the model thinks, a reply comes back. LangChain standardizes the shape of the request and the reply so that every model, on every provider, behaves the same way to your code.

```mermaid
sequenceDiagram
    participant You as Your code
    participant LC as LangChain
    participant BR as Bedrock
    You->>LC: llm.invoke(messages)
    LC->>BR: Converse API request
    BR-->>LC: model reply
    LC-->>You: AIMessage with content_blocks
```

In [2]:
reply = llm.invoke([
    ("system", "You are TravelMind, a concise airline support assistant."),
    ("human", "Passenger Rao, PNR JX48Q2, Gold tier. The BLR to DEL leg was cancelled. What are the next steps?"),
])

print(reply.text())

# Passenger Support Summary

**Passenger:** Rao (Gold tier)  
**PNR:** JX48Q2  
**Issue:** BLR-DEL leg cancelled

## Immediate Next Steps:

1. **Rebooking Options**
   - Offer next available flight on same route
   - Alternative routing via connecting cities
   - Gold tier: Priority rebooking + waived change fees

2. **Compensation & Care**
   - Meal vouchers (if applicable wait time)
   - Hotel accommodation (if overnight delay)
   - Ground transportation

3. **Communication**
   - Confirm contact details
   - Send rebooking confirmation + new itinerary
   - Provide cancellation reference

4. **Tier Benefits Applied**
   - Gold tier: Priority queue, complimentary upgrades if available
   - Loyalty points: Consider goodwill addition

## Required Information:
- Passenger's preferred rebooking option?
- Onward connections affected?
- Any special requirements?

Would you like me to help with specific rebooking options or compensation calculations?


/workspace/ - Strands/.venv/lib/python3.14/site-packages/IPython/core/interactiveshell.py:3748: LangChainDeprecationWarning: Calling .text() as a method is deprecated. Use .text as a property instead (e.g., message.text).
  exec(code_obj, self.user_global_ns, self.user_ns)


### Walkthrough

| Piece | What it does | Why it matters |
|---|---|---|
| `("system", ...)` | sets the assistant role and standing rules | steers every reply without repeating yourself |
| `("human", ...)` | the user turn | the actual request |
| `llm.invoke([...])` | one synchronous call, one reply | the atom; everything else composes this |
| `reply.text()` | pulls the plain text out of the reply | convenience over reading raw blocks |

**Runtime behavior.** One network round trip to Bedrock. `temperature=0.2` keeps the reply steady across runs, which you want for support flows where surprises are bugs.

**Production note.** In a service you rarely call `invoke` bare. You wrap it in a prompt template and a parser so inputs stay clean and outputs stay typed. That is the rest of this lab.

## Messages and roles: the real input

You never send a raw string to a chat model. You send a list of role-tagged messages. Three roles cover almost everything.

| Role | Purpose | TravelMind example |
|---|---|---|
| system | standing instructions and persona | "You are TravelMind, calm and concise." |
| human | the user turn | "Rebook JX48Q2 to the morning flight." |
| ai | a prior reply replayed as context | "Done. New segment AI302, 07:10." |

The tuple form `("human", "text")` is shorthand. When you want objects, use the explicit message classes.

In [3]:
from langchain_core.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage("You are TravelMind, a concise airline support assistant."),
    HumanMessage("Passenger Rao asks: can I get a lounge pass for the delay on JX48Q2?"),
]

print(llm.invoke(messages).text())

I'd be happy to help with Rao's lounge pass request for flight JX48Q2.

To assist you better, I need a few details:

1. **Delay duration** - How long was the delay?
2. **Ticket class** - What cabin class was the booking (Economy, Business, etc.)?
3. **Airline policy** - Does your airline automatically provide lounge passes for delays, or is this discretionary?

**General guidance:** Most airlines offer lounge access for delays of 2+ hours, though policies vary. Business/First class passengers often have better eligibility.

Could you provide these details so I can give you a specific answer?


/workspace/ - Strands/.venv/lib/python3.14/site-packages/IPython/core/interactiveshell.py:3748: LangChainDeprecationWarning: Calling .text() as a method is deprecated. Use .text as a property instead (e.g., message.text).
  exec(code_obj, self.user_global_ns, self.user_ns)


## content_blocks: the reply is structured, not a blob

Older LangChain handed you one opaque string. Version 1 replies carry typed parts. A plain answer is a single text block. A tool-using reply also carries `tool_call` blocks. A reasoning model can carry `reasoning` blocks.

```mermaid
graph LR
    R["AIMessage.content_blocks"] --> T["text"]
    R --> RE["reasoning"]
    R --> TC["tool_call"]
    R --> C["citation"]
```

Reading blocks instead of the raw string is what lets one code path handle a chat answer, a tool request, and a cited answer without special-casing each of them.

In [4]:
reply = llm.invoke([
    ("human", "In one sentence, what is a Bedrock inference profile?"),
])

for block in reply.content_blocks:
    print(block.get("type"), "->", block)

text -> {'type': 'text', 'text': 'A Bedrock inference profile is an AWS configuration that optimizes model performance and cost by routing requests to the most suitable foundation model or model variant based on your specified criteria.'}


## Prompt templates: stop gluing strings

An f-string mixes your logic with your prompt text and forgets about roles. A template separates the two, understands roles, and clicks into the pipe you meet next.

| Concern | f-string | ChatPromptTemplate |
|---|---|---|
| Roles | you hand-build them | built in |
| Reuse | copy and paste | one object, many inputs |
| Composition | manual | slots into the pipe |
| Variable safety | your problem | variables are explicit |

In [5]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are {persona}. Keep replies under 3 sentences."),
    ("human", "{question}"),
])

filled = prompt.invoke({
    "persona": "TravelMind, an airline support assistant",
    "question": "Is seat 14C a window on JX48Q2?",
})
print(filled.to_messages())

[SystemMessage(content='You are TravelMind, an airline support assistant. Keep replies under 3 sentences.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Is seat 14C a window on JX48Q2?', additional_kwargs={}, response_metadata={})]


## The pipe: compose steps into one object

The pipe operator wires runnables so the output of each feeds the next. Same instinct as a shell pipe, where `cat log | grep ERROR | sort` flows left to right.

```mermaid
graph LR
    IN["input dict"] --> P["prompt template"]
    P --> M["chat model"]
    M --> PA["output parser"]
    PA --> OUT["clean text"]
```

The result of piping is itself a runnable, so it exposes the same three ways to run: one input, many inputs, or streamed.

In [6]:
from langchain_core.output_parsers import StrOutputParser

chain = prompt | llm | StrOutputParser()

# one input, one result
print(chain.invoke({"persona": "TravelMind", "question": "What is the baggage limit for Gold tier?"}))
print("---")

# many inputs, run concurrently
batch = chain.batch([
    {"persona": "TravelMind", "question": "Is 14C a window or aisle?"},
    {"persona": "TravelMind", "question": "Can I add a meal to JX48Q2?"},
])
print(batch)
print("---")

# streamed, token by token
for chunk in chain.stream({"persona": "TravelMind", "question": "Summarize Gold tier perks."}):
    print(chunk, end="")

I don't have specific information about your airline's Gold tier baggage allowance in my current context. Could you let me know which airline you're asking about, or check your membership details online or contact their customer service for the most accurate baggage limits?
---
["I don't have access to specific seat maps or flight information to tell you whether seat 14C is a window or aisle seat. This depends on the aircraft type and airline configuration—you can check your booking confirmation, airline website, or seat map tool to see the exact layout for your flight.", "I'd be happy to help you add a meal to booking JX48Q2! However, I need a bit more information—are you looking to add a meal to a flight, hotel stay, or tour package? Once you clarify, I can guide you through the process or let you know what options are available."]
---
I don't have specific information about a "Gold tier" system you're referring to. Could you provide more context about what program or service this is

## Structured output: get objects, not prose

Prose is for humans. Code needs fields. `with_structured_output` binds a schema to the model and hands you a validated Python object. No regex, no hoping.

**Decision matrix: do I need structured output?**

| Situation | Structured output |
|---|---|
| Output feeds another function or a database | Yes |
| You need the same fields on every call | Yes |
| A free-form reply to a human | No |
| One-off exploration in a notebook | No |

In [7]:
from pydantic import BaseModel, Field

class Rebooking(BaseModel):
    pnr: str = Field(description="the booking reference")
    new_flight: str = Field(description="the new flight number")
    fare_difference: float = Field(description="extra fare in USD, 0 if none")

extractor = llm.with_structured_output(Rebooking)

result = extractor.invoke(
    "Rebooked JX48Q2 onto AI302 for a fare difference of 45 dollars."
)
print(result)
print(type(result).__name__, "->", result.new_flight, result.fare_difference)

pnr='JX48Q2' new_flight='AI302' fare_difference=45.0
Rebooking -> AI302 45.0


### Walkthrough

| Line | What it does | Why |
|---|---|---|
| `class Rebooking(BaseModel)` | declares the target shape | one source of truth for the fields |
| `Field(description=...)` | tells the model what each field means | better extraction, fewer misses |
| `llm.with_structured_output(Rebooking)` | binds the schema to the model | the model now returns this type |
| `result.new_flight` | attribute access on a real object | no string parsing, anywhere |

**Runtime behavior.** In version 1 this rides inside the model call, so a structured extraction is one round trip, not two.

**Scenarios.** Pull a `Rebooking` from a messy agent transcript. Turn a support email into a typed `Ticket`. Force a classifier to return one of a fixed set of labels.

**Production use.** Structured output is the seam between the model and the rest of your system. A database write, an API call, a UI render: each wants typed data. This is where prose becomes safe to pass on.

## Portability: swap the model, keep the code

`init_chat_model` is a provider-agnostic front door. Change one argument to move between Bedrock, Anthropic direct, OpenAI, or Gemini. The prompts, parsers, and pipes downstream never notice.

| Provider | model_provider |
|---|---|
| Anthropic on Bedrock | `bedrock_converse` |
| Anthropic direct | `anthropic` |
| OpenAI | `openai` |
| Google Gemini | `google_genai` |

In [8]:
from langchain.chat_models import init_chat_model

portable = init_chat_model(
    MODEL_ID,
    model_provider="bedrock_converse",
    region_name=REGION,
)

# same downstream chain, different engine underneath
print((prompt | portable | StrOutputParser()).invoke(
    {"persona": "TravelMind", "question": "One line: what is Gold tier priority boarding?"}
))

Gold tier priority boarding allows you to board flights before most passengers, typically after first/business class, giving you better overhead bin access and seat selection flexibility.


## Recap: the atom, mastered

```mermaid
graph LR
    D["prompt template"] --> A["messages"]
    A --> B["chat model"]
    B --> C["content_blocks"]
    C --> E["structured output"]
    B --> F["pipe: invoke, batch, stream"]
```

**Reach-for-it card**

| You want | Reach for |
|---|---|
| A model reply | `ChatBedrockConverse` |
| Role-tagged input | messages, tuple or class form |
| Reusable, parameterized prompts | `ChatPromptTemplate` |
| Typed objects out | `with_structured_output` |
| Composed steps, run three ways | the pipe plus `invoke`, `batch`, `stream` |
| A one-line vendor switch | `init_chat_model` |

Lab 2 keeps every one of these and arranges them into workflows you fully control: chaining, routing, parallelization, orchestrator-workers, and evaluator-optimizer.

## Exercises

Small, fast, grounded in what you just ran. No new APIs required.

1. **A typed classifier.** Write a Pydantic model `Intent` with one field `category`, and in its description restrict it to `rebooking`, `refund`, `baggage`, or `other`. Bind it with `with_structured_output` and classify three sample requests.

2. **A three-step pipe.** Build `prompt | llm | StrOutputParser()` that takes `{"pnr": ...}` and returns a one-line status for TravelMind. Run it with `invoke`, then `batch` on three PNRs.

3. **Blocks, not text.** Call the model on any question and print only the `text` blocks from `content_blocks`. Confirm the joined result matches `reply.text()`.

4. **Skeptic's drill.** Take exercise 2 and force a failure: pass a prompt variable the template does not declare. Read the error. Knowing what the template guards against is half the reason to use one.